In [41]:
import os
import matplotlib.pyplot as plt
import numpy as np
import scipy.integrate as sp
import math as mt
from scipy.integrate import solve_ivp

output_dir = "/home/max/Documents/private/Uni/masterthesis/paper/plots"
os.makedirs(output_dir, exist_ok=True)

In [ ]:
def Gamma(d): #Define Gamma function as numerical integral
    result, _ = sp.quad(lambda t: t**(d-1)*np.exp(-t), 0, np.inf) #returns only numerical value, _ "deletes" the error value from sp.quad
    return result

In [43]:
#coupled deq derived from wetterich eq
def flow(k, S, d, rho): #del_t Gamma_k = ...
    mq , l = S #mq = m², l = lambda

    prefactor1 = (12*k**(d+1))/((4*np.pi)**(d/2)*Gamma(d/2 + 1))
    prefactor2 = (2*k**(d+1))/((4*np.pi)**(d/2)*Gamma(d/2 + 1))
 
    dldk = (prefactor1*((2*l**2) / (k**2 +mq + l*rho)**3 + (18*l**2) / (k**2 +mq + 3*l*rho)**3)
      )  #dlambda/dk
    
    dmqdk = (prefactor2*(-l / (k**2 + mq + l*rho)**2 - 3*l / (k**2 + mq + 3*l*rho)**2 ) - rho*dldk/6
      )  #dm²/dk

    return [dmqdk, dldk]

In [ ]:
#initial conditions
d = 3.0
rho = 50.0
k0 = 1000.0
k_end = 0.0

l0_list=[0.0001, 0.001, 0.01, 0.1, 1, 10, 20, 100]
mq0_list=[-16, 16]

k_eval = np.linspace(k0, k_end, 1000)


In [49]:
# Schleife über die beiden m0²-Werte
for mq0 in mq0_list:
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(9, 7), sharex=True)
    
    # Schleife über alle lambda0-Werte
    for l0 in l0_list:
        S0 = [mq0, l0]
        sol = solve_ivp(flow, t_span=(k0, k_end), y0=S0, t_eval=k_eval, method='RK45', args=(d, rho))
        
        if sol.success:
            # m^2(k) linear plotten
            ax1.plot(sol.t, sol.y[0], lw=1.5, label=r'$\lambda_0 = ' + str(l0) + r'$')
            
            # lambda(k) plotten (Werte <= 0 filtern, da log-Skala keine negativen Werte erlaubt)
            l_vals = sol.y[1]
            l_vals_positive = np.where(l_vals > 0, l_vals, np.nan)
            ax2.plot(sol.t, l_vals_positive, lw=1.5, label=r'$\lambda_0 = ' + str(l0) + r'$')
        else:
            print(f"Integration fehlgeschlagen für mq0={mq0}, l0={l0}: {sol.message}")

    # Oberer Plot: m^2(k)
    ax1.set_ylabel(r'$m^2(k)$')
    ax1.grid(True, which='both', ls='-')
    ax1.legend(loc='upper left', fontsize=8, ncol=2)

    # Unterer Plot: lambda(k) mit logarithmischer Y-Achse
    ax2.set_yscale('log')
    ax2.set_ylabel(r'$\lambda(k)$ (log Scale)')
    ax2.set_xlabel(r'Scale $k$ (MeV)')
    ax2.grid(True, which='both', ls='--')
    ax2.legend(loc='upper left', fontsize=8, ncol=2)

    # Zentrale Parameter-Box
    param_text = (
        f"Parameter:\n"
        f"$d = {d}$\n"
        f"$\\rho = \\phi^2 = {rho}$ MeV\n"
        f"$m_0^2 = {mq0}$ MeV²\n"
        f"$k \\in [{k0}, {k_end}]$ MeV"
    )
    ax1.text(0.95, 0.05, param_text, transform=ax1.transAxes, fontsize=9,
             horizontalalignment='right', verticalalignment='bottom',
             bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

    ax1.set_xlim(k0, k_end)
    plt.tight_layout()

    # Speichern je nach mq0-Wert
    file_name = f"rg_flow_mq0_{int(mq0)}_loglambda.png"
    file_path = os.path.join(output_dir, file_name)
    plt.savefig(file_path, dpi=300)
    plt.close(fig)
    
    print(f"Plot gespeichert unter: {file_path}")

Plot gespeichert unter: /home/max/Documents/private/Uni/masterthesis/paper/plots/rg_flow_mq0_-16_loglambda.png
Plot gespeichert unter: /home/max/Documents/private/Uni/masterthesis/paper/plots/rg_flow_mq0_16_loglambda.png


In [48]:
# Schleife über die beiden m0²-Werte
for mq0 in mq0_list:
    # 2x4 Gitter für die 8 l0-Werte
    fig, axes = plt.subplots(2, 4, figsize=(16, 8), sharex=True)
    axes_flat = axes.flatten()
    
    for idx, l0 in enumerate(l0_list):
        ax = axes_flat[idx]
        S0 = [mq0, l0]
        sol = solve_ivp(flow, t_span=(k0, k_end), y0=S0, t_eval=k_eval, method='RK45', args=(d, rho))
        
        if sol.success:
            # m^2(k) auf linker Achse
            ax.plot(sol.t, sol.y[0], color='tab:blue', lw=1.5, label=r'$m^2(k)$')
            ax.set_ylabel(r'$m^2(k)$', color='tab:blue', fontsize=9)
            ax.tick_params(axis='y', labelcolor='tab:blue')
            
            # lambda(k) auf rechter Achse (twinx)
            ax_right = ax.twinx()
            ax_right.plot(sol.t, sol.y[1], color='tab:orange', lw=1.5, label=r'$\lambda(k)$')
            ax_right.set_ylabel(r'$\lambda(k)$', color='tab:orange', fontsize=9)
            ax_right.tick_params(axis='y', labelcolor='tab:orange')
            
            ax.set_title(f"$\\lambda_0 = {l0}$", fontsize=10, fontweight='bold')
            ax.grid(True, alpha=0.3)
            ax.set_xlim(k0, k_end)
        else:
            ax.text(0.5, 0.5, "Integration failed", transform=ax.transAxes, ha='center')

    # X-Achsenbeschriftung für die untere Reihe hinzufügen
    for ax in axes[1, :]:
        ax.set_xlabel(r'Scale $k$ (MeV)', fontsize=9)

    # Gesamttitel für das Bild
    fig.suptitle(f"RG-Flow for $m_0^2 = {mq0}$ MeV² ($d={d}, \\rho={rho}$)", fontsize=14)
    plt.tight_layout()

    # Speichern
    file_name = f"rg_flow_grid_mq0_{int(mq0)}.png"
    file_path = os.path.join(output_dir, file_name)
    plt.savefig(file_path, dpi=300)
    plt.close(fig)
    
    print(f"Kachel-Plot gespeichert unter: {file_path}")

Kachel-Plot gespeichert unter: /home/max/Documents/private/Uni/masterthesis/paper/plots/rg_flow_grid_mq0_-16.png
Kachel-Plot gespeichert unter: /home/max/Documents/private/Uni/masterthesis/paper/plots/rg_flow_grid_mq0_16.png
